# Reading data

#### Reading CSV

In [0]:
display(dbutils.fs.ls('/Volumes/workspace/default/firstvolume/'))

In [0]:
df = spark.read.format('csv').option("header", True).option("inferSchema", True).load('/Volumes/workspace/default/firstvolume/BigMart Sales.csv')

In [0]:
df.show()

In [0]:
df.display()

#### Reading JSON file

###### multiLine
- True - for multi line json file
- False - for single line json file

In [0]:
df_json = spark.read.option("header", True)\
                    .option("inferSchema", True)\
                    .option("multiLine", False)\
                    .json('/Volumes/workspace/default/firstvolume/drivers.json')
df_json.show()

In [0]:
df_json.display()

### Schema Definition

In [0]:
df.printSchema()

In [0]:
my_ddl_schema = '''
  Item_Identifier string,
  Item_Weight string,
  Item_Fat_Content string,
  Item_Visibility double,
  Item_Type string,
  Item_MRP double,
  Outlet_Identifier string,
  Outlet_Establishment_Year int,
  Outlet_Size string,
  Outlet_Location_Type string,
  Outlet_Type string,
  Item_Outlet_Sales double'''

In [0]:
df = spark.read.schema(my_ddl_schema)\
          .option("header", True)\
          .csv('/Volumes/workspace/default/firstvolume/BigMart Sales.csv')
df.printSchema()

### StructType Schema

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
my_structtype_schema = StructType([
                                    StructField('Item_Identifier', StringType(), True),
                                    StructField('Item_Weight', IntegerType(), True),
                                    StructField('Item_Fat_Content', StringType(), True),
                                    StructField('Item_Visibility', DoubleType(), True),
                                    StructField('Item_Type', StringType(), True),
                                    StructField('Item_MRP', DoubleType(), True),
                                    StructField('Outlet_Identifier', StringType(), True),
                                    StructField('Outlet_Establishment_Year', IntegerType(), True),
                                    StructField('Outlet_Size', StringType(), True),
                                    StructField('Outlet_Location_Type', StringType(), True),
                                    StructField('Outlet_Type', StringType(), True),
                                    StructField('Item_Outlet_Sales', DoubleType(), True)
                                  ])

In [0]:
df = spark.read.schema(my_structtype_schema)\
            .option('header', True)\
            .format('csv')\
            .load('/Volumes/workspace/default/firstvolume/BigMart Sales.csv')
df.printSchema()

In [0]:
df = spark.read.csv('/Volumes/workspace/default/firstvolume/BigMart Sales.csv', header=True, inferSchema=True)
df.printSchema()

## SELECT

In [0]:
df.select("Item_Identifier", "Item_Weight", "Item_Fat_Content").show()

In [0]:
df.select(col("Item_Identifier"), col("Item_Weight"), col("Item_Fat_Content")).display()

## Alias

In [0]:
df.select(col("Item_Identifier").alias('Item_ID')).display()

## Filter/WHERE
1) Filter the data with fat content = Regular
2) Slice the data with item type = Soft Drinks and weight < 10
3) Fetch the data with Tier in (Tier1 or Tier 2) and Outlet Size is Null

In [0]:
df.filter(col("Item_Fat_Content")=="Regular").display()

In [0]:
df.filter((col("Item_Type")=="Soft Drinks")&(col("Item_Weight")<10)).display()

In [0]:
df.filter((col("Outlet_Location_Type").isin("Tier 1", "Tier 2"))&(col("Outlet_Size").isNull())).display()

## Renaming columns

In [0]:
df.withColumnRenamed("Item_Weight","Item_Wt").display()

### WithColumn - to add new column, to modify existing column
**Add new column - flag**

In [0]:
df.withColumn("flag", lit("new")).display()

In [0]:
df.withColumn("Item_Calculation", col("Item_Weight")*col("Item_MRP")).show()

In [0]:
df.withColumn("Item_Fat_Content", regexp_replace(col("Item_Fat_Content"),"Regular","Reg"))\
    .withColumn("Item_Fat_Content", regexp_replace(col("Item_Fat_Content"),"Low Fat", "LF"))\
    .show()

In [0]:
df.withColumn("Item_Fat_Content", when(col("Item_Fat_Content")=="Regular","Reg").when(col("Item_Fat_Content")=="Low Fat", "LF").otherwise(col("Item_Fat_Content"))).show()

### Type Casting
Convert a type of the data

In [0]:
df.withColumn("Item_Weight", col("Item_Weight").cast(StringType())).display()

## Sort/OrderBy
- .sort() - .asc()/.desc()

In [0]:
df.sort(col("Item_Weight").desc()).display()

In [0]:
df.sort(col("Item_Visibility").asc()).display()

**Sorting in multiple columns**

In [0]:
df.sort(["Item_Weight", "Item_Visibility"], ascending=[0,0]).show()

In [0]:
df.sort(["Item_Weight", "Item_Visibility"], ascending=[0,1]).show()

## Limit
Like 'Top' in TSQL

In [0]:
df.limit(10).display()

## Drop
To drop/delete a single column or multiple columns

In [0]:
df.drop("Item_Weight").show()

In [0]:
df.drop("Item_Weight", "Item_Identifier").show()

## drop_duplicates/dropDuplicates
To drop the duplicate rows or columns

In [0]:
df.dropDuplicates().show()

In [0]:
df.dropDuplicates(subset=["Item_Type"]).show()

## Distinct

In [0]:
df.distinct().show()